# 04 — 预训练对比: From-Scratch vs HuggingFace Trainer

本 notebook 对比 **手写训练循环** 和 **HF Trainer** 两种预训练方式的核心差异。

## 对比维度

| 维度 | From-Scratch | HuggingFace |
|------|-------------|-------------|
| 训练配置 | `PreTrainer.__init__` 手动参数 | `TrainingArguments` 一站式配置 |
| Labels 构建 | 手动 `input_ids[1:]` 右移 | `DataCollatorForLanguageModeling` 自动处理 |
| 训练循环 | `while step < max_steps` 手写 | `Trainer.train()` 一行搞定 |
| 梯度累积 | 手动计数 + loss 缩放 | `gradient_accumulation_steps` 参数 |
| 混合精度 | `GradScaler` + `amp_autocast` | `bf16=True` / `fp16=True` |
| 学习率调度 | `CosineWarmupScheduler` 手写 | `lr_scheduler_type="cosine"` |
| 梯度裁剪 | `clip_grad_norm` 手动调用 | `max_grad_norm` 参数 |
| 验证 | `evaluate_loss()` 手写循环 | `Trainer` 内置 evaluation |
| 早停 | `EarlyStopping` 手写类 | `EarlyStoppingCallback` |
| 日志 | `TrainingLogger` 手写 | `ClearMindLoggingCallback` |
| Checkpoint | `torch.save` / `torch.load` | `save_pretrained` / `from_pretrained` |

## 1. 训练配置对比

### From-Scratch: 手动传参

```python
# from-scratch 版: PreTrainer 构造函数接收大量参数
trainer = PreTrainer(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    lr=3e-4,
    batch_size=16,
    gradient_accumulation=4,
    max_steps=10000,
    warmup_steps=500,
    eval_every=500,
    save_every=1000,
    patience=5,
    dtype="float32",
    log_every=10,
    output_dir="outputs/pretrain",
)
```

### HuggingFace: TrainingArguments 一站式

```python
# HF 版: YAML 配置直接透传为 TrainingArguments
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="outputs/pretrain",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,   # 有效 batch = 16
    max_steps=200,
    learning_rate=5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    logging_steps=10,
    bf16=False,
    report_to="none",
)
```

**核心差异:**
- From-scratch 需要手动实现 `CosineWarmupScheduler`、`GradScaler`、`EarlyStopping` 等组件
- HF 版通过 `TrainingArguments` 的参数名直接配置，Trainer 内部自动创建对应组件

## 2. DataCollator vs 手动 Labels

### From-Scratch: 手动构建 labels

```python
# from-scratch 版: PretrainDataset.__getitem__
def __getitem__(self, idx):
    chunk = self.chunks[idx]  # 已 tokenize 的固定长度 chunk
    input_ids = chunk[:-1]    # 去掉最后一个 token
    labels = chunk[1:]        # 去掉第一个 token (右移一位)
    return {"input_ids": input_ids, "labels": labels}
```

### HuggingFace: DataCollator 自动处理

```python
# HF 版: DataCollatorForLanguageModeling 自动生成 labels
from transformers import DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# collator 内部逻辑:
# 1. 动态 padding 到 batch 中最长序列
# 2. labels = input_ids.clone()
# 3. labels[padding_positions] = -100  (忽略 padding 的 loss)
# 4. Trainer 的 compute_loss 中自动右移
```

**核心差异:**
- From-scratch 在 Dataset 中预计算 `input_ids[:-1]` 和 `labels[1:]`
- HF 版的 `DataCollator` 动态处理 padding，模型内部 `forward()` 自动右移计算 loss

## 3. 训练循环对比

### From-Scratch: 手写 while loop (~50 行)

```python
# from-scratch 版: PreTrainer.train() 核心循环
step = start_step
while step < self.max_steps:
    # 梯度累积内循环
    for micro in range(self.gradient_accumulation):
        batch = next(data_iter)
        input_ids = batch["input_ids"].to(self.device)
        labels = batch["labels"].to(self.device)
        
        with amp_autocast(self.device, self.dtype):
            logits, loss, _ = self.model(input_ids, labels)
        
        scaled_loss = loss / self.gradient_accumulation
        self.scaler.scale(scaled_loss).backward()
        total_loss += loss.item()
    
    # 梯度裁剪 + 优化器步进 + 调度器步进
    grad_norm, lr = self._optimizer_step()
    step += 1
    
    # 日志、验证、保存 checkpoint
    if step % self.log_every == 0: ...
    if step % self.eval_every == 0: ...
    if step % self.save_every == 0: ...
```

### HuggingFace: 一行搞定

```python
# HF 版: Trainer 封装了完整的训练循环
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["validation"],
    data_collator=collator,
    callbacks=[ClearMindLoggingCallback()],
)

trainer.train()  # 一行代码，内部实现了上面所有逻辑
trainer.save_model(output_dir)
```

**核心差异:**
- From-scratch 需要手动管理：数据迭代器重建、梯度累积计数、`StopIteration` 处理、device 转移
- HF Trainer 内部自动处理一切，通过 `TrainingArguments` 和 `Callback` 提供等效的控制力

## 4. Callback vs TrainingLogger

### From-Scratch: TrainingLogger 类

```python
# from-scratch 版: 手写日志系统
class TrainingLogger:
    def log(self, step, loss, lr, grad_norm, tokens_per_sec):
        elapsed = time.time() - self.start_time
        eta = elapsed / step * (self.max_steps - step)
        print(f"step {step:>6d} | loss={loss:.4f} | lr={lr:.2e} | "
              f"grad_norm={grad_norm:.4f} | tokens/s={tokens_per_sec:.0f} | "
              f"ETA={format_time(eta)}")
    
    def save_log(self, path):
        # 保存 JSONL 格式的训练日志
        ...
```

### HuggingFace: TrainerCallback 机制

```python
# HF 版: 继承 TrainerCallback，覆盖生命周期方法
class ClearMindLoggingCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        # 打印训练配置摘要
        ...
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        # 格式化输出 step/loss/lr/grad_norm
        ...
    
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        # 打印验证结果
        ...
    
    def on_train_end(self, args, state, control, **kwargs):
        # 打印训练总结
        ...
```

**核心差异:**
- From-scratch 的 `TrainingLogger` 在 `train()` 循环中手动调用
- HF 的 `TrainerCallback` 通过事件驱动自动触发，解耦了日志逻辑和训练循环

## 5. Checkpoint 保存对比

### From-Scratch: `torch.save`

```python
# from-scratch 版: 手动序列化所有训练状态
def save_checkpoint(path, model, optimizer, scheduler_step, step, loss):
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_step": scheduler_step,
        "step": step,
        "loss": loss,
    }, path)

# 加载: 手动恢复每个组件
checkpoint = torch.load(path)
model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
```

### HuggingFace: `save_pretrained` / `from_pretrained`

```python
# HF 版: 标准化的模型保存格式
trainer.save_model("outputs/pretrain")
# 生成文件:
#   config.json          — 模型配置
#   model.safetensors    — 模型权重 (安全格式)
#   tokenizer.json       — tokenizer 配置

# 加载: 一行代码
model = ClearMindForCausalLM.from_pretrained("outputs/pretrain")
```

**核心差异:**
- From-scratch 用 `.pth` 包含 model + optimizer + scheduler 状态
- HF 用标准 `safetensors` 格式，可直接上传 HuggingFace Hub，也可被其他框架加载

## 总结

| 方面 | From-Scratch 优势 | HuggingFace 优势 |
|------|-------------------|-------------------|
| 学习价值 | 深入理解训练循环每一步 | 理解工业级框架设计模式 |
| 灵活性 | 完全自定义每个细节 | 通过 Callback 扩展 |
| 开发效率 | 较低，需写大量样板代码 | 极高，几十行完成训练 |
| 生态兼容 | 需自行处理 | 与 HF Hub、accelerate 无缝集成 |
| 调试难度 | 代码简单，易理解 | 框架内部较复杂 |